# 04. Translation Dataset & DataLoader Pipeline

This notebook tests and validates the numericalization and batching pipeline:
1. Loads saved `eng_vocab.json` and `amh_vocab.json`.
2. Tests `sentence_to_ids` and `pad_sequence_ids`.
3. Instantiates `TranslationDataset` for train, validation, and test splits.
4. Builds PyTorch `DataLoader` with `BATCH_SIZE = 64`.
5. Verifies batch shapes, padding consistency, and token alignment.


In [ ]:
# Environment & Path Setup
# If running on Google Colab, uncomment the lines below:
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/english-amharic-nmt

import os
import sys
from pathlib import Path

# Add project root to sys.path
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.paths import get_data_paths
from src.utils.seed import set_seed

# Configure data directory (can be overridden by DATA_ROOT environment variable)
# On Colab: DATA_ROOT = "/content/drive/MyDrive/english-amharic-nmt-data"
DATA_ROOT = os.getenv("DATA_ROOT", str(PROJECT_ROOT / "data"))
paths = get_data_paths(DATA_ROOT)
paths.ensure_directories()
set_seed(42)

print("Project root:", PROJECT_ROOT)
print("Data directory:", paths.data_root)


## 1. Load Vocabularies

In [ ]:
from src.data.vocabulary import load_vocab, PAD_IDX, UNK_IDX, SOS_IDX, EOS_IDX

eng_vocab = load_vocab(paths.eng_vocab_path)
amh_vocab = load_vocab(paths.amh_vocab_path)

print(f"English vocab: {len(eng_vocab):,} tokens")
print(f"Amharic vocab: {len(amh_vocab):,} tokens")


## 2. Test Sentence-to-IDs and Padding Functions

In [ ]:
from src.data.vocabulary import sentence_to_ids, pad_sequence_ids

sample_en = "What is the second woe recorded by Habakkuk ?"
sample_am = "ዕንባቆም የመዘገበው ሁለተኛው ወዮታ ምንድን ነው ?"

en_ids = sentence_to_ids(sample_en, eng_vocab)
am_ids = sentence_to_ids(sample_am, amh_vocab)

print("English IDs (with SOS/EOS):", en_ids)
print("Amharic IDs (with SOS/EOS):", am_ids)

en_padded = pad_sequence_ids(en_ids, max_len=20)
print("\nPadded English IDs (len=20):", en_padded)


## 3. Instantiate TranslationDataset
We load the filtered datasets and wrap them in PyTorch `TranslationDataset` instances.

In [ ]:
import pandas as pd
from src.data.dataset import TranslationDataset

train_df = pd.read_csv(paths.train_filtered_path)
val_df = pd.read_csv(paths.val_filtered_path)
test_df = pd.read_csv(paths.test_filtered_path)

train_dataset = TranslationDataset(train_df, eng_vocab, amh_vocab, max_len=70)
val_dataset = TranslationDataset(val_df, eng_vocab, amh_vocab, max_len=70)
test_dataset = TranslationDataset(test_df, eng_vocab, amh_vocab, max_len=70)

print(f"Train dataset:      {len(train_dataset):,} samples")
print(f"Validation dataset: {len(val_dataset):,} samples")
print(f"Test dataset:       {len(test_dataset):,} samples")

src_sample, trg_sample = train_dataset[0]
print(f"Sample tensor shapes: src={src_sample.shape}, trg={trg_sample.shape}, dtype={src_sample.dtype}")


## 4. Build PyTorch DataLoaders (BATCH_SIZE = 64)

In [ ]:
from src.data.dataset import get_dataloader

BATCH_SIZE = 64

train_loader = get_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = get_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = get_dataloader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches:      {len(train_loader):,}")
print(f"Validation batches: {len(val_loader):,}")
print(f"Test batches:       {len(test_loader):,}")


## 5. Verify Batch Properties & Padding

In [ ]:
src_batch, trg_batch = next(iter(train_loader))

print(f"Batch shapes: Source={src_batch.shape}, Target={trg_batch.shape}")
print(f"Source PAD count: {(src_batch == PAD_IDX).sum().item():,}")
print(f"Target PAD count: {(trg_batch == PAD_IDX).sum().item():,}")

# Verify EOS and PAD positioning for first sample
src_0 = src_batch[0]
eos_pos = (src_0 == EOS_IDX).nonzero(as_tuple=True)[0]
pad_pos = (src_0 == PAD_IDX).nonzero(as_tuple=True)[0]

print(f"Sample 0 EOS token position: {eos_pos[0].item() if len(eos_pos) > 0 else 'N/A'}")
print(f"Sample 0 First PAD position: {pad_pos[0].item() if len(pad_pos) > 0 else 'No PAD'}")
